# Прогноз покупки банковских продуктов по транзакциям

Все эксперименты из README по порядку. Нужен Colab с GPU (Среда выполнения, Сменить среду, T4), на всё уходит около 3 часов.
Результаты пишутся в `runs/`, их стоит копировать на Google Диск, иначе пропадут при отключении сессии.

In [ ]:
import os
if not os.path.exists("/content/life-events"):
    !git clone -q https://github.com/RomanPhys/life-events.git /content/life-events
%cd /content/life-events
!pip -q install lightgbm huggingface_hub pyarrow
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = "/content/drive/MyDrive/life-events-runs"

## Тесты

In [ ]:
!python -m unittest discover -s tests -t . -v 2>&1 | tail -25

## Синтетика

В `target_2` сигнал есть только в порядке событий (тип 7, потом тип 8 в течение двух дней).
Бустинг на агрегатах должен видеть его плохо, а последовательные модели хорошо. Если это не так, значит где-то баг.

In [ ]:
!python -m src.synthetic --out data/synth --clients 3000
!python -m src.prepare --raw data/synth --out prepared/synth
!python -m src.gbm --data prepared/synth --name synth_gbm --runs runs_synth
!python -m src.train finetune --data prepared/synth --model transformer --name synth_tr --runs runs_synth --epochs 8
!python -m src.train finetune --data prepared/synth --model gru --name synth_gru --runs runs_synth --epochs 8
!python -m src.report --runs runs_synth --out reports/synthetic.md

## Данные MBD-mini

Скачиваю только транзакции, таргеты и разбиение (около 2 ГБ) и смотрю, что внутри.
Таргеты должны идти примерно на месяц позже транзакций.

In [ ]:
!python -m src.inspect_mbd --download --root data/mbd

In [ ]:
# так считались результаты в README; поправка часового пояса проверяется в конце ноутбука
!python -m src.prepare --raw data/mbd --out prepared/mbd --max-clients 40000 --last-months 6

## LightGBM и проба утечки

`gbm_leak` видит события целевого месяца. Если он заметно лучше обычного, значит ошибка в отсечке дала бы завышенные результаты.

In [ ]:
!python -m src.gbm --data prepared/mbd --name gbm
!python -m src.gbm --data prepared/mbd --name gbm_leak --leak

## GRU и трансформер с нуля, 3 сида

In [ ]:
for s in [0, 1, 2]:
    !python -m src.train finetune --data prepared/mbd --model gru --name gru_s{s} --seed {s} 2>&1 | tee -a train.log
    !python -m src.train finetune --data prepared/mbd --model transformer --name tr_scratch_s{s} --seed {s} 2>&1 | tee -a train.log

## MLM-предобучение и дообучение

Предобучение только на клиентах из обучающей части, метки не используются. Валидационный MLM-лосс должен падать.

In [ ]:
!python -m src.train pretrain --data prepared/mbd --name mlm --steps 6000 2>&1 | tee -a train.log
for s in [0, 1, 2]:
    !python -m src.train finetune --data prepared/mbd --model transformer --init runs/mlm/encoder.pt --lr 1e-4 --name tr_mlm_s{s} --seed {s} 2>&1 | tee -a train.log

## Мало разметки: 5% и 20% клиентов

In [ ]:
for frac in [0.05, 0.2]:
    for s in [0, 1]:
        tag = str(frac).replace('.', 'p')
        !python -m src.train finetune --data prepared/mbd --model transformer --label-frac {frac} --name tr_scratch_f{tag}_s{s} --seed {s} 2>&1 | tee -a train.log
        !python -m src.train finetune --data prepared/mbd --model transformer --init runs/mlm/encoder.pt --lr 1e-4 --label-frac {frac} --name tr_mlm_f{tag}_s{s} --seed {s} 2>&1 | tee -a train.log

## Эмбеддинги из предобученного энкодера в бустинге

In [ ]:
!python -m src.train embed --data prepared/mbd --ckpt runs/mlm/encoder.pt --out runs/mlm/embeddings.npz
!python -m src.gbm --data prepared/mbd --name gbm_emb_only --emb runs/mlm/embeddings.npz --emb-only
!python -m src.gbm --data prepared/mbd --name gbm_plus_emb --emb runs/mlm/embeddings.npz

## Итоговая таблица и парные сравнения

Сравнение идёт на одном тесте, бутстрэп по клиентам. Если указан шаблон с `*`, предсказания сидов усредняются.

In [ ]:
!python -m src.report --runs runs --out reports/results.md
for a, b in [("runs/tr_scratch_s*", "runs/tr_mlm_s*"),
             ("runs/gbm", "runs/tr_mlm_s*"),
             ("runs/gbm", "runs/gru_s*"),
             ("runs/gru_s*", "runs/tr_mlm_s*"),
             ("runs/tr_scratch_f0p05_s*", "runs/tr_mlm_f0p05_s*"),
             ("runs/tr_scratch_f0p2_s*", "runs/tr_mlm_f0p2_s*"),
             ("runs/gbm", "runs/gbm_plus_emb")]:
    !python -m src.report --compare "{a}" "{b}" --data prepared/mbd 2>&1 | tee -a reports/compare.txt

## Часовой пояс

Время в MBD записано в UTC, а месяцы меток московские. Пересчитываю LightGBM со сдвигом на 3 часа и сравниваю с исходным.

In [ ]:
!python -m src.prepare --raw data/mbd --out prepared/mbd_msk --max-clients 40000 --last-months 6 --time-shift-hours 3 > /dev/null
!python -m src.gbm --data prepared/mbd_msk --name gbm_msk --runs runs_msk 2>&1 | tee reports/timezone_check.txt
!python -m src.report --compare runs/gbm runs_msk/gbm_msk --data prepared/mbd 2>&1 | tee -a reports/timezone_check.txt

## Сохранение

In [ ]:
!mkdir -p "$SAVE" && cp -r runs runs_msk reports train.log "$SAVE"/
!rm -rf results && mkdir -p results
!for d in runs/*/ runs_msk/*/; do n=$(basename $d); mkdir -p results/$n; cp $d/metrics.json results/$n/ 2>/dev/null; cp $d/pretrain_log.json results/$n/ 2>/dev/null; done
!cp -r reports results/ && zip -qr results.zip results
from google.colab import files
files.download("results.zip")